# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print fundamental metadata (name & description). Access metadata fields as attributes, not by subscript.
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(dataset.metadata, 'version', 'N/A')}")
print(f"License: {getattr(dataset.metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's find out what record sets are present in the dataset, and for each record set, list their fields and columns by their `@id` as per Croissant schema.

In [ ]:
record_sets = dataset.record_sets
print("Record Sets found:")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name} | @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (id: {field.id}, dtype: {getattr(field,'data_type',None)})")
    if hasattr(rs, 'columns'):
        print("  Columns:")
        for column in rs.columns:
            print(f"    - {column.name} (id: {column.id}, dtype: {getattr(column,'data_type',None)})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll extract one or more record sets of interest.

In [ ]:
# Find the record set IDs to extract; replace with those found in the overview
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    if df.shape[1] > 0:
        dataframes[record_set_id] = df
        print(f"Fields: {df.columns.tolist()}")
        print(df.head(2).to_string(index=False))
    else:
        print("No tabular data found in this record set.")
    print("-"*40 + "\n")
# For demonstration, pick the first available record_set with data
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"We'll use record set: {main_record_set_id} for further exploration.")
else:
    main_record_set_id = None
    print("No tabular record set found!")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
import numpy as np

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]

    # Identify numeric fields automatically for demonstration
    numeric_fields = df.select_dtypes(include=np.number).columns.tolist()
    if len(numeric_fields) == 0:
        # Try to convert 'likely numeric' columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        numeric_fields = df.select_dtypes(include=np.number).columns.tolist()

    if numeric_fields:
        # Pick the first numeric field
        numeric_field = numeric_fields[0]
        print(f"Analyzing numeric field '@id': {numeric_field}")

        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")
        print(filtered_df.head(2))

        # Normalize the numeric field (z-score)
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, normalized_col]].head(2))

        # Try to group by a categorical/text field if available
        group_fields = df.select_dtypes(include=[object]).columns.tolist()
        group_field = None
        for gf in group_fields:
            # Only consider if not too unique
            if df[gf].nunique() > 1 and df[gf].nunique() < len(df)//2:
                group_field = gf
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped by '{group_field}' mean of {numeric_field}:")
            print(grouped_df.head(5))
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("Did not find numeric fields for automatic EDA. Please select one manually.")
else:
    print("No data loaded to explore.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field and (if possible) its relationship with a group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_fields:
    plt.figure(figsize=(7,5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric fields or data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we used the FAIR\u2072 Croissant schema and `mlcroissant` to programmatically load the dataset and inspect its basic metadata.
* We enumerated available record sets and their corresponding field `@id` values.
* A primary record set was loaded into a DataFrame. We explored a key numeric data field, applied filtering and normalization, and visualized its distribution.
* This workflow makes it easy to further automate downstream normalization, analysis, or harmonization for mass spectrometry proteomics datasets.

Refer to the [dataset documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/latest/) to extend this analysis.